### Representación básica: RFM

| Variable    | Cálculo                                                            |
| ----------- | ------------------------------------------------------------------ |
| `recency`   | días desde la última compra del cliente hasta el final del periodo |
| `frequency` | número de días distintos en los que el cliente realizó compras     |
| `monetary`  | suma del precio de todos los artículos comprados                   |


In [1]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../../data/hym/")

In [2]:
transactions = pd.read_csv(DATA_DIR / "transactions_train.csv")


transactions["t_dat"] = pd.to_datetime(transactions["t_dat"])

### Recency

Utilizamos como fecha de referencia el día posterior a la última fecha disponible en los datos

In [3]:


reference_date = transactions["t_dat"].max() + pd.Timedelta(days=1)

recency = (
    transactions
    .groupby("customer_id")["t_dat"]
    .max()
    .reset_index()
)

recency["recency"] = (
    reference_date - recency["t_dat"]
).dt.days

recency = recency[["customer_id", "recency"]]

recency.head()

,customer_id,recency
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,18
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,77
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,8
3,00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...,472
4,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,42


### Frequency

Como no tenemos identificador de pedido, contamos días de compra distintos.

In [4]:
frequency = (
    transactions
    .groupby("customer_id")["t_dat"]
    .nunique()
    .reset_index(name="frequency")
)

frequency.head()

,customer_id,frequency
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,10
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,23
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,7
3,00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...,1
4,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,6


### Monetary

In [5]:
monetary = (
    transactions
    .groupby("customer_id")["price"]
    .sum()
    .reset_index(name="monetary")
)

monetary.head()

,customer_id,monetary
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0.648983
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,2.601932
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,0.704780
3,00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...,0.060983
4,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,0.469695


### Unión de las tres variables al dataset

In [6]:
hm_simple = (
    recency
    .merge(frequency, on="customer_id", how="inner")
    .merge(monetary, on="customer_id", how="inner")
)

hm_simple.head()

,customer_id,recency,frequency,monetary
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,18,10,0.648983
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,77,23,2.601932
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,8,7,0.704780
3,00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...,472,1,0.060983
4,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,42,6,0.469695


In [7]:
print("Dimensiones:", hm_simple.shape)

print("\nClientes duplicados:")
print(hm_simple["customer_id"].duplicated().sum())

print("\nValores nulos:")
print(hm_simple.isnull().sum())

print("\nEstadísticas:")
display(hm_simple[["recency", "frequency", "monetary"]].describe())

Dimensiones: (1362281, 4)

Clientes duplicados:
0

Valores nulos:
customer_id    0
recency        0
frequency      0
monetary       0
dtype: int64

Estadísticas:


,recency,frequency,monetary
count,1.362281e+06,1.362281e+06,1.362281e+06
mean,2.361484e+02,6.665423e+00,6.493858e-01
std,2.211188e+02,9.707005e+00,1.197582e+00
min,1.000000e+00,1.000000e+00,7.627119e-04
25%,4.900000e+01,1.000000e+00,8.806780e-02
50%,1.520000e+02,3.000000e+00,2.455932e-01
75%,3.990000e+02,8.000000e+00,7.011864e-01
max,7.340000e+02,4.270000e+02,5.767641e+01


In [8]:
print("Clientes en transactions:")
print(transactions["customer_id"].nunique())

print("\nClientes en hm_simple:")
print(hm_simple["customer_id"].nunique())

Clientes en transactions:
1362281

Clientes en hm_simple:
1362281


### Guardado del dataset básico RFM

In [9]:
hm_simple.to_csv(
    "../../simple_datasets/hym_model_simple.csv",
    index=False
)